# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

- **Title:** Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells
- **Description:** This dataset details the analysis of protein abundance, modifications, and sequences in human (Homo sapiens) samples. It includes various fields such as accession, description, coverage percentage, peptide counts, and molecular weight (MW) alongside calculated parameters such as pI and normalized abundances across different samples.
- **DOI:** 10.71728/senscience.vq4a-28xa

_You will explore the dataset using references by `@id` for all entities._

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset Name: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"DOI: {getattr(metadata, 'identifier', 'N/A')}")
if hasattr(metadata, 'keywords'):
    print(f"Keywords: {', '.join(metadata.keywords)}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We'll list all record sets, fields, and columns in the dataset by their `@id`. This is crucial, as all data access will reference entities by their `@id` field.

In [ ]:
def print_record_sets_and_fields(ds):
    print("\nRecord sets available in the dataset:\n")
    record_sets = getattr(ds.metadata, 'recordSets', None)
    # For croissant 1.0, it may be found as 'recordSet' in the metadata dict
    if not record_sets and hasattr(ds.metadata, 'recordSet'):
        record_sets = getattr(ds.metadata, 'recordSet', [])
    if not record_sets:
        print("No record sets found in metadata.")
    else:
        for rs in record_sets:
            rs_id = getattr(rs, '@id', None) or rs.get('@id')
            rs_name = getattr(rs, 'name', None) or rs.get('name', '')
            print(f"- Record set: {rs_name} (@id: {rs_id})")
            # Fields may be under 'fields' or 'field'
            fields = getattr(rs, 'fields', None) or getattr(rs, 'field', None) or rs.get('fields', []) or rs.get('field', [])
            if not fields:
                print("  [No fields listed]")
            else:
                print("  Fields:")
                for f in fields:
                    f_id = getattr(f, '@id', None) or f.get('@id', '')
                    f_name = getattr(f, 'name', None) or f.get('name', '')
                    print(f"    - {f_name} (@id: {f_id})")

print_record_sets_and_fields(dataset)

# For interactive exploration, we can also print out one record set's first few records by @id:
print("\nExample records from the first found record set (referenced by @id):")
record_sets = getattr(metadata, 'recordSet', [])
if len(record_sets):
    rs_id = getattr(record_sets[0], '@id', None) or record_sets[0].get('@id')
    for i, rec in enumerate(dataset.records(record_set=rs_id)):
        print(rec)
        if i > 2:
            break

## 3. Data Extraction
Load data from specific record set(s) into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

For this dataset, we will extract the first available record set (by `@id`).

In [ ]:
# Helper: Extract data from each record set
record_sets = getattr(metadata, 'recordSet', [])
if not record_sets:
    print("No record sets available in metadata.")
else:
    # If record_sets appear as a dict, put in list
    if isinstance(record_sets, dict):
        record_sets = [record_sets]
    # Build list of all record set @ids
    record_sets_ids = [getattr(rs, '@id', None) or rs.get('@id') for rs in record_sets]
    dataframes = {}
    for rs_id in record_sets_ids:
        print(f"Loading records for record set: {rs_id}")
        records = list(dataset.records(record_set=rs_id))
        if not records:
            print(f"No records found for record set {rs_id}.")
            continue
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded {len(df)} records for {rs_id}.")
    # Show columns for the first record set as an example
    first_rs_id = record_sets_ids[0]
    print(f"\nColumns for record set {first_rs_id}:\n{dataframes[first_rs_id].columns.tolist()}")
    dataframes[first_rs_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

We'll select a numeric field (e.g., coverage, MW, or peptide count) by its column label (`@id` or actual column name), filter for values above a threshold, normalize them, and optionally group by a categorical field.

In [ ]:
# Choose the first record set (you can change as needed)
rs_id = list(dataframes.keys())[0]
df = dataframes[rs_id].copy()

# Try to select a common numeric field -- for this dataset, field names might be like 'coverage', 'MW', etc.
# Let's auto-detect a likely numeric field (float or int columns)
possible_numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
if not possible_numeric_fields:
    # Try to coerce object columns to numeric if possible
    for col in df.columns:
        try:
            df[col] = pd.to_numeric(df[col])
        except Exception:
            continue
    possible_numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]

if not possible_numeric_fields:
    print("No numeric fields found.")
else:
    numeric_field = possible_numeric_fields[0]
    print(f"Using numeric field '{numeric_field}' for filtering and normalization.")

    # Filter records above a threshold
    threshold = df[numeric_field].mean()  # use mean as a simple threshold
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold:.3f} (mean): {len(filtered_df)} records")

    # Normalize the numeric field (z-score)
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try grouping by a categorical field (other than the numeric_field)
    possible_group_fields = [col for col in df.columns if col != numeric_field and pd.api.types.is_object_dtype(df[col])]
    if possible_group_fields:
        group_field = possible_group_fields[0]
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"\nMean of {numeric_field} grouped by {group_field} (first 5 groups):")
        print(grouped_df.head())
    else:
        group_field = None

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Below we plot the distribution of the selected numeric field and a boxplot by group if grouping is available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if 'numeric_field' in locals():
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=30, color='skyblue')
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.show()

    if 'group_field' in locals() and group_field is not None:
        plt.figure(figsize=(10,4))
        # Only show up to 10 groups for clarity
        top_groups = df[group_field].value_counts().index[:10]
        sns.boxplot(data=df[df[group_field].isin(top_groups)], x=group_field, y=numeric_field)
        plt.title(f'{numeric_field} across {group_field} (top 10 groups)')
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

## 6. Conclusion
In this notebook, we:
- Loaded the FAIR<sup>2</sup> dataset Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells via its Croissant schema (referenced by URL)
- Explored available record sets, fields, and columns referencing them by their `@id`
- Loaded records programmatically into DataFrames for processing
- Performed exploratory analysis by filtering, normalizing, and grouping records
- Visualized the distribution and groupwise behavior of key numeric measures

This workflow can be easily adapted for additional Croissant datasets using their `@id` for robust and reproducible data science pipelines.